# CENG493 — Turkish Legal RAG: Colab Setup (A100)

## Prerequisites — read before running anything

### 1. Set the runtime to A100 GPU + High RAM
Go to **Runtime → Change runtime type**, set:
- **Hardware accelerator**: A100 GPU
- **Runtime shape**: High-RAM

Then click **Save** and reconnect. The A100 has 80 GB VRAM, which allows batch size 64 for embeddings
and running `qwen2.5:14b` via Ollama.

### 2. Files required in Google Drive
Place the following files inside `My Drive/CENG493_Datasets/`:

| File | Description |
|---|---|
| `corpus.jsonl` | Pre-chunked corpus in evaluator format |
| `rag_eval.json` | RAG evaluation questions with gold chunk IDs |
| `gold_benchmark.json` | HMGS-style gold benchmark questions |
| `embedding.jsonl` | Embedding fine-tuning triplets (optional) |
| `reranker.jsonl` | Reranker training data (optional) |
| `llm.jsonl` | LLM fine-tuning data (optional) |

### 3. Notebook structure
| Step | What | Required? |
|---|---|---|
| 1 | Verify A100 GPU + Mount Drive + copy datasets | Yes |
| 2 | Clone repo (`main` branch) | Yes |
| 3 | Install Python dependencies (faiss-gpu for A100) | Yes |
| 4 | Patch config for A100 (batch size 64, qwen2.5:14b) | Yes |
| 5 | Build FAISS index from corpus.jsonl | Yes |
| 6 | Retrieval evaluation (dense + RRF+Rerank, no LLM) | Yes |
| 7 | Note about optional QA generation | — |
| 8 | Install and start Ollama + pull qwen2.5:14b | For QA eval |
| 9 | Full QA evaluation (generation + scoring) | For QA eval |
| 10 | Note about optional fine-tuning | — |
| 11 | Embedding fine-tuning (BGE-M3 on Turkish legal triplets) | Optional |
| 12 | LLM fine-tuning (Qwen2.5-3B with QLoRA) | Optional |
| 13 | Export LoRA adapter to Ollama (qwen25-legal-ft) | Optional |
| 14 | Full ablation evaluation (all stages via script 14) | Optional |
| 15 | Save results + models + index to Google Drive | Optional |

In [ ]:
# =============================================================================
# Cell 1 — GPU verification + Mount Google Drive + Copy datasets
#
# Verifies we are on an A100, mounts Drive, and copies the six dataset files
# from My Drive/CENG493_Datasets/ to /content/datasets/ for fast local access.
#
# If DRIVE_DIR points to the wrong folder, update it below before running.
# =============================================================================

import subprocess
import pathlib
import shutil
from google.colab import drive

# ── GPU check ────────────────────────────────────────────────────────────────
gpu_info = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']
).decode().strip()
print(f"GPU: {gpu_info}")
if 'A100' not in gpu_info:
    print()
    print("WARNING: Not on A100. The notebook is optimised for A100 (80 GB VRAM).")
    print("         Go to Runtime → Change runtime type → A100 GPU + High RAM.")
else:
    print("A100 confirmed. Proceeding.")

# ── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive')

# ── Paths — update DRIVE_DIR if your files are in a different folder ──────────
DRIVE_DIR      = pathlib.Path('/content/drive/MyDrive/CENG493_Datasets')
LOCAL_DATASETS = pathlib.Path('/content/datasets')
LOCAL_DATASETS.mkdir(parents=True, exist_ok=True)

DATASET_FILES = [
    'corpus.jsonl',       # pre-chunked corpus (required for index + retrieval)
    'rag_eval.json',      # RAG eval questions with gold chunk IDs (required for retrieval eval)
    'gold_benchmark.json',# HMGS-style gold benchmark (required for retrieval eval)
    'embedding.jsonl',    # embedding fine-tuning triplets (optional)
    'reranker.jsonl',     # reranker training data (optional)
    'llm.jsonl',          # LLM fine-tuning data (optional)
]

print()
missing_required = []
for fname in DATASET_FILES:
    src = DRIVE_DIR / fname
    dst = LOCAL_DATASETS / fname
    if not src.exists():
        tag = 'REQUIRED' if fname in ('corpus.jsonl', 'rag_eval.json', 'gold_benchmark.json') else 'optional'
        print(f"  MISSING [{tag}]: {src}  ← upload this to Drive first")
        if tag == 'REQUIRED':
            missing_required.append(fname)
        continue
    shutil.copy(src, dst)
    size_mb = dst.stat().st_size / 1e6
    print(f"  Copied  {fname}  ({size_mb:.1f} MB)")

print()
if missing_required:
    raise FileNotFoundError(
        f"Required dataset files not found in Drive: {missing_required}\n"
        f"Upload them to {DRIVE_DIR} and re-run this cell."
    )
print("Datasets ready at", LOCAL_DATASETS)

In [ ]:
# =============================================================================
# Cell 2 — Clone the repository (main branch)
#
# Clones the Turkish Legal RAG repo into /content/CENG493_Project and
# changes the working directory so all subsequent !python calls resolve
# relative paths correctly.
# =============================================================================

!git clone --branch main --single-branch \
    https://github.com/BeratMert29/Turkish-Legal-Question-Answering-RAG.git \
    /content/CENG493_Project
%cd /content/CENG493_Project/CENG493_Project
import os
print("Working directory:", os.getcwd())

In [ ]:
# =============================================================================
# Cell 3 — Install Python dependencies
#
# Key difference from a T4 setup:
#   • faiss-gpu-cu12 (not faiss-cpu) — GPU-accelerated FAISS on A100 (CUDA 12)
#   • All other packages are the same as requirements-colab.txt
#
# If Colab already has faiss-cpu from a previous session, we uninstall it first
# to avoid conflicts.
# =============================================================================

# Remove faiss-cpu if installed (conflicts with faiss-gpu)
!pip uninstall -q -y faiss-cpu 2>/dev/null || true

# GPU-accelerated FAISS for CUDA 12 (A100)
!pip install -q faiss-gpu-cu12

# Core RAG pipeline dependencies
!pip install -q \
    sentence-transformers==3.4.1 \
    transformers==4.57.6 \
    openai==1.63.2 \
    rouge-score==0.1.2 \
    nltk==3.9.1 \
    httpx \
    "rank_bm25>=0.2.2" \
    "ranx>=0.3.5" \
    "langchain-text-splitters>=0.3.0" \
    "scipy>=1.12.0" \
    "evaluate>=0.4.0" \
    "tqdm>=4.66.0"

# Fine-tuning dependencies (needed for Cells 11–12; safe to install now)
!pip install -q \
    bitsandbytes \
    "peft>=0.7.0" \
    "trl>=0.7.0" \
    "datasets>=2.14.0"

print()
print("Dependencies installed.")
print("If you see 'RESTART RUNTIME' banner — click it, then re-run from Cell 1.")

In [ ]:
# =============================================================================
# Cell 4 — Patch config for A100
#
# The repo's config.py ships conservative defaults (EMBEDDING_BATCH_SIZE=8,
# LLM_MODEL='qwen2.5:14b') that target a T4 with 16 GB VRAM.
# On A100 (80 GB VRAM) we raise the batch size to 64 for much faster indexing
# and confirm the 14b model is configured.
#
# This cell MUST run every time the kernel restarts, before any other code.
# =============================================================================

import sys
sys.path.insert(0, '/content/CENG493_Project/CENG493_Project')

import config

# A100 has 80 GB VRAM — batch size 64 fills it efficiently for BGE-M3
config.EMBEDDING_BATCH_SIZE = 64

# Use the full 14b parameter model (A100 can handle it; T4 would use 7b)
config.LLM_MODEL = "qwen2.5:14b"
config.LLM_JUDGE_MODEL = "llama3.3:70b"

# Ensure output directories exist
config.INDEX_DIR.mkdir(parents=True, exist_ok=True)
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Config patched for A100:")
print(f"  EMBEDDING_MODEL      : {config.EMBEDDING_MODEL}")
print(f"  EMBEDDING_BATCH_SIZE : {config.EMBEDDING_BATCH_SIZE}")
print(f"  LLM_MODEL            : {config.LLM_MODEL}")
print(f"  LLM_JUDGE_MODEL      : {config.LLM_JUDGE_MODEL}")
print(f"  RERANKER_MODEL       : {config.RERANKER_MODEL}")
print(f"  TOP_K_RETRIEVAL      : {config.TOP_K_RETRIEVAL}")
print(f"  RERANKER_CANDIDATES  : {config.RERANKER_CANDIDATES}")
print(f"  INDEX_DIR            : {config.INDEX_DIR}")
print(f"  PROCESSED_DIR        : {config.PROCESSED_DIR}")

In [ ]:
# =============================================================================
# Cell 5 — Build FAISS index from corpus.jsonl
#
# Runs scripts/02_build_index.py with --corpus pointing to the local copy of
# corpus.jsonl. The script:
#   1. Normalises the evaluator JSONL format to the project's internal schema
#   2. Encodes all chunks with BAAI/bge-m3 (batch size 64 on A100 ≈ 5–10 min)
#   3. Writes faiss.index + metadata.jsonl to config.INDEX_DIR
#
# Re-running is safe (idempotent) — it overwrites the index files.
# =============================================================================

!PYTHONUTF8=1 python scripts/02_build_index.py --corpus /content/datasets/corpus.jsonl

In [ ]:
# =============================================================================
# Cell 6 — Retrieval evaluation (new dataset format, no LLM required)
#
# Evaluates retrieval quality on both rag_eval.json and gold_benchmark.json
# using inline Python (not 03_evaluate_retrieval.py, which reads the old CSV).
#
# Retrieval modes evaluated:
#   • Dense          — FAISS top-k with BGE-M3 embeddings
#   • RRF + Rerank   — Reciprocal Rank Fusion (dense+BM25), then BGE reranker
#
# Metrics reported per dataset:
#   Recall@5, Recall@10, MRR, nDCG@10
#
# Outputs are also stored in Python variables for use in Cell 9 (QA eval).
# =============================================================================

import json
import time
import sys
from pathlib import Path

sys.path.insert(0, '/content/CENG493_Project/CENG493_Project')

# Re-apply A100 patch in case kernel restarted
import config
config.EMBEDDING_BATCH_SIZE = 64
config.LLM_MODEL = 'qwen2.5:14b'

from data.corpus_loader import load_corpus_jsonl
from data.data_processor import DataProcessor, CorpusChunk
from retrieval.embedder import Embedder
from retrieval.retriever import Retriever
from retrieval.bm25_retriever import BM25Index
from retrieval.reranker import Reranker
from evaluation.retrieval_metrics import compute_all_metrics

# ── Load and normalise corpus ─────────────────────────────────────────────────
print("Loading corpus...")
raw_chunks = load_corpus_jsonl(Path('/content/datasets/corpus.jsonl'))
corpus_chunks = [
    CorpusChunk(**{k: r[k] for k in ('chunk_id', 'doc_id', 'text', 'source', 'char_len')})
    for r in raw_chunks
]
print(f"  {len(corpus_chunks):,} chunks loaded")

# ── Load RAG eval questions ───────────────────────────────────────────────────
print("Loading rag_eval.json...")
with open('/content/datasets/rag_eval.json', encoding='utf-8') as f:
    rag_eval_raw = json.load(f)
qa_dicts = [{
    'query_id'      : item['query_id'],
    'question'      : item['query'],
    'answer'        : item.get('gold_answer_extract', ''),
    'context'       : '',
    'source'        : item.get('source', ''),
    'data_type'     : '',
    'gold_source_ids': item.get('gold_chunk_ids', []),
} for item in rag_eval_raw]
print(f"  {len(qa_dicts):,} RAG eval questions loaded")

# ── Load gold benchmark (HMGS-style) ─────────────────────────────────────────
print("Loading gold_benchmark.json...")
with open('/content/datasets/gold_benchmark.json', encoding='utf-8') as f:
    gold_data = json.load(f)
gold_dicts = [{
    'query_id'      : item['question_id'],
    'question'      : item['question'],
    'answer'        : item.get('verified_answer', ''),
    'context'       : '',
    'source'        : (
        item['gold_sources'][0].get('source', '')
        if item.get('gold_sources') else ''
    ),
    'data_type'     : '',
    'gold_source_ids': [s['source_id'] for s in item.get('gold_sources', [])],
} for item in gold_data]
print(f"  {len(gold_dicts):,} gold benchmark questions loaded")

# ── Load FAISS index ──────────────────────────────────────────────────────────
print("\nLoading FAISS index + embedding model...")
embedder = Embedder()
embedder.load_model()
retriever = Retriever(embedder)
retriever.load_index(
    config.INDEX_DIR / config.INDEX_FILE,
    config.INDEX_DIR / config.METADATA_FILE,
)
print(f"  Index: {retriever.index.ntotal:,} vectors")

# ── Build relevance maps ──────────────────────────────────────────────────────
# build_relevant_chunk_map uses gold_source_ids from the qa_dicts (Strategy 0)
print("\nBuilding relevance maps...")
rel_map_eval = DataProcessor.build_relevant_chunk_map(corpus_chunks, qa_dicts)
rel_map_gold = DataProcessor.build_relevant_chunk_map(corpus_chunks, gold_dicts)
print(f"  eval rel_map: {len(rel_map_eval)} queries with relevant chunks")
print(f"  gold rel_map: {len(rel_map_gold)} queries with relevant chunks")

# ── Build BM25 index ──────────────────────────────────────────────────────────
print("\nBuilding BM25 index...")
t0 = time.time()
bm25 = BM25Index()
bm25.build([{'text': c.text, 'chunk_id': c.chunk_id} for c in corpus_chunks])
print(f"  BM25 built in {time.time() - t0:.1f}s")

# ── Load reranker ─────────────────────────────────────────────────────────────
print("\nLoading reranker (BAAI/bge-reranker-v2-m3)...")
reranker = Reranker()
reranker.load_model()
print("  Reranker ready.")

# ── Evaluation helper ─────────────────────────────────────────────────────────
def evaluate_retrieval(qa_items, rel_map, label):
    """Run dense and RRF+Rerank retrieval, compute metrics, print table."""
    questions = [q['question'] for q in qa_items]
    print(f"\n{'='*60}")
    print(f"  {label}  ({len(questions)} queries)")
    print(f"{'='*60}")

    # Dense retrieval
    t0 = time.time()
    dense_results = retriever.batch_retrieve(questions, top_k=config.TOP_K_RETRIEVAL)
    print(f"  Dense retrieval      : {time.time() - t0:.1f}s")

    # RRF fusion of dense + BM25, then reranker
    t0 = time.time()
    rrf_candidates = retriever.batch_rrf_retrieve(
        questions, bm25, top_k=config.RERANKER_CANDIDATES
    )
    rrf_reranked = reranker.batch_rerank(
        questions, rrf_candidates, top_k=config.TOP_K_RETRIEVAL
    )
    print(f"  RRF + rerank         : {time.time() - t0:.1f}s")

    def build_metric_input(all_retrieved):
        return [
            {
                'query_id' : q['query_id'],
                'relevant' : rel_map.get(q['query_id'], []),
                'retrieved': [c['chunk_id'] for c in chunks],
            }
            for q, chunks in zip(qa_items, all_retrieved)
        ]

    m_dense = compute_all_metrics(build_metric_input(dense_results))
    m_rrf   = compute_all_metrics(build_metric_input(rrf_reranked))

    header = f"  {'Mode':<20}  {'R@5':>8}  {'R@10':>8}  {'MRR':>8}  {'nDCG@10':>8}"
    print(header)
    print(f"  {'-'*62}")
    for name, m in [("Dense", m_dense), ("RRF+Rerank", m_rrf)]:
        print(
            f"  {name:<20}  "
            f"{m.get('recall_at_5', 0):8.4f}  "
            f"{m.get('recall_at_10', 0):8.4f}  "
            f"{m.get('mrr', 0):8.4f}  "
            f"{m.get('ndcg_at_10', 0):8.4f}"
        )

    return dense_results, rrf_reranked, m_dense, m_rrf

# ── Run evaluations ───────────────────────────────────────────────────────────
eval_dense, eval_rrf, eval_m_dense, eval_m_rrf = evaluate_retrieval(
    qa_dicts, rel_map_eval, "RAG Eval Set (rag_eval.json)"
)
gold_dense, gold_rrf, gold_m_dense, gold_m_rrf = evaluate_retrieval(
    gold_dicts, rel_map_gold, "Gold Benchmark (gold_benchmark.json)"
)

print()
print("Retrieval evaluation complete.")
print("Variables available for Cell 9 (QA eval):")
print("  corpus_chunks, qa_dicts, gold_dicts, retriever, bm25, reranker")
print("  eval_rrf  — RRF+reranked results for rag_eval questions")

## Optional: QA Generation with Ollama

Cells 8 and 9 install Ollama, pull `qwen2.5:14b`, and run full generation + QA scoring.

**Skip Cells 8–9 if you only need retrieval metrics** (Cell 6 above is sufficient).

**A100 note:** `qwen2.5:14b` runs on the A100 GPU via Ollama. Expect ~5–15 s per query.
Generating answers for a few hundred questions typically takes 30–60 minutes.
Make sure your Colab session won’t time out before it finishes.

In [ ]:
# =============================================================================
# Cell 8 — Install Ollama + start server + pull qwen2.5:14b  [OPTIONAL]
#
# Steps:
#   1. Install Ollama via the official install script
#   2. Start the Ollama server as a background daemon
#   3. Pull qwen2.5:14b (≈10 GB download; takes 5–15 min depending on Colab bandwidth)
#
# Skip this cell and Cell 9 if you only need retrieval metrics.
# =============================================================================

import subprocess
import time

# Install Ollama
print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

# Start the Ollama server (background daemon)
print("\nStarting Ollama server...")
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(8)  # allow server to initialise before pulling
print("Ollama server started (PID", proc.pid, ")")

# Pull the model (≈10 GB; progress shown inline)
print("\nPulling qwen2.5:14b (this may take several minutes)...")
!ollama pull qwen2.5:14b

# Pull judge model (separate from generator to avoid self-evaluation bias)
print("\nPulling llama3.3:70b (LLM judge model, ≈40 GB)...")
!ollama pull llama3.3:70b

# Quick sanity check — both qwen2.5:14b and llama3.3:70b must appear
print("\nVerifying models are available...")
!ollama list
print("\nBoth models ready. Proceed to Cell 9.")

In [ ]:
# =============================================================================
# Cell 9 — Full QA evaluation: generation + scoring  [OPTIONAL]
#
# Prerequisites:
#   • Cell 6 must have run (corpus_chunks, qa_dicts, retriever, bm25, reranker
#     must be in scope; eval_rrf must be available for re-use or recomputed here)
#   • Cell 8 must have run (Ollama running with qwen2.5:14b)
#
# Pipeline:
#   1. Re-uses RRF+reranked retrieval results from Cell 6 (no extra retrieval cost)
#   2. For each question, assembles context from top-k chunks and generates an answer
#   3. Injects [Kaynak N] citation markers via token-overlap heuristic
#   4. Scores with ROUGE, BERTScore, citation accuracy, and hallucination analysis
#   5. Saves predictions.jsonl + qa_metrics.json to results/colab_eval/
# =============================================================================

import json
import sys
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, '/content/CENG493_Project/CENG493_Project')
import config
config.EMBEDDING_BATCH_SIZE = 64
config.LLM_MODEL = 'qwen2.5:14b'

from generation.rag_pipeline import RAGPipeline
from evaluation.qa_metrics import compute_all_qa_metrics_with_citation
from utils import inject_citations, check_ollama

# ── Validate Ollama is running ────────────────────────────────────────────────
if not check_ollama(config.LLM_BASE_URL, config.LLM_MODEL):
    raise RuntimeError(
        "Ollama is not running or qwen2.5:14b is not available.\n"
        "Run Cell 8 first, then re-run this cell."
    )
print(f"Ollama OK: {config.LLM_MODEL} is available.")

# ── Check that retrieval results from Cell 6 are in scope ────────────────────
try:
    _ = eval_rrf
    _ = qa_dicts
    _ = retriever
    _ = bm25
    _ = reranker
    print("Retrieval results from Cell 6 are in scope.")
except NameError:
    print("WARNING: Cell 6 variables not found. Re-running retrieval...")
    # Minimal re-import if kernel restarted mid-session
    from data.corpus_loader import load_corpus_jsonl
    from data.data_processor import DataProcessor, CorpusChunk
    from retrieval.embedder import Embedder
    from retrieval.retriever import Retriever
    from retrieval.bm25_retriever import BM25Index
    from retrieval.reranker import Reranker
    import time

    raw_chunks = load_corpus_jsonl(Path('/content/datasets/corpus.jsonl'))
    corpus_chunks = [
        CorpusChunk(**{k: r[k] for k in ('chunk_id','doc_id','text','source','char_len')})
        for r in raw_chunks
    ]
    with open('/content/datasets/rag_eval.json', encoding='utf-8') as f:
        rag_eval_raw = json.load(f)
    qa_dicts = [{
        'query_id'      : item['query_id'],
        'question'      : item['query'],
        'answer'        : item.get('gold_answer_extract', ''),
        'context'       : '',
        'source'        : item.get('source', ''),
        'data_type'     : '',
        'gold_source_ids': item.get('gold_chunk_ids', []),
    } for item in rag_eval_raw]

    embedder = Embedder(); embedder.load_model()
    retriever = Retriever(embedder)
    retriever.load_index(
        config.INDEX_DIR / config.INDEX_FILE,
        config.INDEX_DIR / config.METADATA_FILE,
    )
    bm25 = BM25Index()
    bm25.build([{'text': c.text, 'chunk_id': c.chunk_id} for c in corpus_chunks])
    reranker = Reranker(); reranker.load_model()

    rrf_candidates = retriever.batch_rrf_retrieve(
        [q['question'] for q in qa_dicts], bm25, top_k=config.RERANKER_CANDIDATES
    )
    eval_rrf = reranker.batch_rerank(
        [q['question'] for q in qa_dicts], rrf_candidates, top_k=config.TOP_K_RETRIEVAL
    )
    print("Retrieval re-run complete.")

# ── Set up RAG pipeline ───────────────────────────────────────────────────────
pipeline = RAGPipeline(retriever)

# ── Generate answers ──────────────────────────────────────────────────────────
print(f"\nGenerating answers for {len(qa_dicts)} questions...")
predictions = []
errors = []

for qa, chunks in tqdm(zip(qa_dicts, eval_rrf), total=len(qa_dicts), desc="Generating"):
    try:
        ctx, ctx_chunks = pipeline.assemble_context(chunks)
        answer = pipeline.generate(qa['question'], ctx)
        answer = inject_citations(answer, ctx_chunks)
        predictions.append({
            'query_id'        : qa['query_id'],
            'question'        : qa['question'],
            'predicted'       : answer,
            'expected'        : qa['answer'],
            'retrieved_sources': [c.get('source', '') if isinstance(c, dict)
                                  else getattr(c, 'source', '') for c in ctx_chunks],
            'expected_source' : qa['source'],
            'retrieved_chunks': [
                c if isinstance(c, dict) else vars(c)
                for c in ctx_chunks
            ],
        })
    except Exception as e:
        errors.append({'query_id': qa['query_id'], 'error': str(e)})
        print(f"  ERROR {qa['query_id']}: {e}")

print(f"Generated {len(predictions)} answers; {len(errors)} errors.")

# ── Score predictions ─────────────────────────────────────────────────────────
print("\nComputing QA metrics...")
qa_metrics = compute_all_qa_metrics_with_citation(predictions)
print("\nQA Metrics:")
for k, v in sorted(qa_metrics.items()):
    if isinstance(v, float):
        print(f"  {k:<40}: {v:.4f}")
    else:
        print(f"  {k:<40}: {v}")

# ── Save results ──────────────────────────────────────────────────────────────
out_dir = config.BASE_DIR / 'results' / 'colab_eval'
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / 'predictions.jsonl', 'w', encoding='utf-8') as f:
    for p in predictions:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

with open(out_dir / 'qa_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(qa_metrics, f, ensure_ascii=False, indent=2)

if errors:
    with open(out_dir / 'errors.json', 'w', encoding='utf-8') as f:
        json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"\nResults saved to {out_dir}")
print("  predictions.jsonl —", len(predictions), "entries")
print("  qa_metrics.json")

## Optional: Fine-Tuning

Cells 11 and 12 fine-tune the embedding model and the LLM respectively.
Both are **optional** and independent of each other.

| Cell | What | Dataset file | Estimated time on A100 |
|---|---|---|---|
| 11 | Fine-tune BGE-M3 (embedding model) | `embedding.jsonl` | 1–4 h |
| 12 | Fine-tune Qwen2.5 LLM with LoRA | `llm.jsonl` | 2–6 h |

Fine-tuning requires the corresponding `.jsonl` files to have been copied
from Drive in Cell 1. If they are missing, those cells will raise an error.

In [ ]:
# =============================================================================
# Cell 11 — Embedding fine-tuning (BGE-M3 on Turkish legal triplets)  [OPTIONAL]
#
# Fine-tunes BAAI/bge-m3 using contrastive learning on the triplets in
# embedding.jsonl (format: {"id": ..., "query": ..., "positive_passage": ...}).
#
# The script reads triplets from config.PROCESSED_DIR/embedding_triplets.jsonl
# so we copy the file there before running.
#
# Fine-tuned weights are saved to config.FINETUNED_EMBEDDING_MODEL
# (models/bge-m3-turkish-legal/).
# =============================================================================

import shutil
import sys
from pathlib import Path

sys.path.insert(0, '/content/CENG493_Project/CENG493_Project')
import config
config.EMBEDDING_BATCH_SIZE = 64

src = Path('/content/datasets/embedding.jsonl')
if not src.exists():
    raise FileNotFoundError(
        "embedding.jsonl not found at /content/datasets/embedding.jsonl\n"
        "Ensure it was uploaded to Drive and Cell 1 completed successfully."
    )

dst = config.PROCESSED_DIR / 'embedding_triplets.jsonl'
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(src, dst)
print(f"Embedding triplets copied: {src} -> {dst}")

# Count triplets for reference
n_triplets = sum(1 for _ in open(dst, encoding='utf-8'))
print(f"Training on {n_triplets:,} triplets")
print(f"Output model: {config.FINETUNED_EMBEDDING_MODEL}")
print()

# Run fine-tuning (A100-optimised: batch_size 64 is set in config above)
!PYTHONUTF8=1 python scripts/12_finetune_embeddings.py

print("\nEmbedding fine-tuning complete.")
print(f"Fine-tuned model saved to: {config.FINETUNED_EMBEDDING_MODEL}")

In [ ]:
# =============================================================================
# Cell 12 — LLM fine-tuning (Qwen2.5-14B-Instruct with LoRA)  [OPTIONAL]
#
# Fine-tunes Qwen2.5 using LoRA / QLoRA via the TRL SFTTrainer.
# Training data is read from config.PROCESSED_DIR/qa_train_merged.jsonl
# (format: instruction-following pairs in llm.jsonl).
#
# After fine-tuning, run scripts/13_export_lora_to_ollama.py to package
# the adapter into an Ollama model named 'qwen25-legal-ft'.
#
# Fine-tuned LoRA adapters are saved under models/qwen25-legal-ft/.
# =============================================================================

import shutil
import sys
from pathlib import Path

sys.path.insert(0, '/content/CENG493_Project/CENG493_Project')
import config

src = Path('/content/datasets/llm.jsonl')
if not src.exists():
    raise FileNotFoundError(
        "llm.jsonl not found at /content/datasets/llm.jsonl\n"
        "Ensure it was uploaded to Drive and Cell 1 completed successfully."
    )

dst = config.PROCESSED_DIR / 'qa_train_merged.jsonl'
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(src, dst)
print(f"LLM training data copied: {src} -> {dst}")

n_examples = sum(1 for _ in open(dst, encoding='utf-8'))
print(f"Training on {n_examples:,} examples")
print()

# Run LLM fine-tuning
!PYTHONUTF8=1 python scripts/08_finetune_llm.py --backend qlora

print("\nLLM fine-tuning complete.")
print("To export LoRA adapters to Ollama format, run:")
print("  !python scripts/13_export_lora_to_ollama.py")

In [ ]:
# =============================================================================
# Cell 13 — Export fine-tuned LoRA adapter to Ollama  [OPTIONAL]
#
# Requires Cell 12 to have completed successfully.
# Steps performed by scripts/13_export_lora_to_ollama.py:
#   1. Merge LoRA adapter into base Qwen2.5-14B weights (merge_and_unload)
#   2. Convert merged model to GGUF (q8_0) via llama.cpp
#   3. Write Ollama Modelfile
#   4. Register model as 'qwen25-legal-ft' with Ollama
#
# After this cell, 'qwen25-legal-ft' will be available for Cell 14 ablation.
# Estimated time on A100: 20–40 minutes (llama.cpp clone + conversion).
# =============================================================================

import sys
from pathlib import Path

sys.path.insert(0, '/content/CENG493_Project/CENG493_Project')
import config
config.EMBEDDING_BATCH_SIZE = 64
config.LLM_MODEL = 'qwen2.5:14b'

adapter_dir = config.BASE_DIR / 'models' / 'qwen25_lora'
if not adapter_dir.exists():
    raise FileNotFoundError(
        f"LoRA adapter not found at {adapter_dir}.\n"
        "Run Cell 12 (LLM fine-tuning) first."
    )

print(f"LoRA adapter found: {adapter_dir}")
print("Starting export pipeline (merge → GGUF → Ollama)...")
print("This will take 20–40 minutes on A100.\n")

!PYTHONUTF8=1 python scripts/13_export_lora_to_ollama.py

print("\nVerifying model registration...")
!ollama list
print("\nDone. 'qwen25-legal-ft' is ready for ablation evaluation in Cell 14.")


In [ ]:
# =============================================================================
# Cell 14 — Full Ablation Evaluation (all pipeline stages)
#
# Runs scripts/14_eval_all_stages.py which evaluates all available pipeline
# configurations and produces a comparative ablation table.
#
# Stages attempted (skipped automatically if prerequisites are missing):
#   base       — BGE-M3 base + dense retrieval + qwen2.5:14b
#   rrf_rerank — BGE-M3 base + RRF+reranker   + qwen2.5:14b  (best retrieval)
#   llm_ft     — BGE-M3 base + dense           + qwen25-legal-ft  (requires Cell 13)
#   emb_ft     — BGE-M3 ft   + RRF+reranker   + qwen2.5:14b      (requires Cell 11)
#   full       — BGE-M3 ft   + RRF+reranker   + qwen25-legal-ft  (requires Cells 11+13)
#
# Dataset: gold_benchmark.json (240 HMGS exam questions, evaluator format)
# Results saved to results/stage_*/baseline_metrics.json + ablation_summary.json
#
# Estimated time on A100: 30–90 min depending on which stages are available.
# =============================================================================

import subprocess
import sys
from pathlib import Path

sys.path.insert(0, '/content/CENG493_Project/CENG493_Project')
import config
config.EMBEDDING_BATCH_SIZE = 64
config.LLM_MODEL = 'qwen2.5:14b'

# ── Dataset paths (from Cell 1) ───────────────────────────────────────────────
CORPUS_PATH    = '/content/datasets/corpus.jsonl'
EVAL_DATA_PATH = '/content/datasets/gold_benchmark.json'

for p in [CORPUS_PATH, EVAL_DATA_PATH]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Dataset not found: {p}\nRe-run Cell 1 to copy datasets from Drive.")

# ── Determine which stages to run ─────────────────────────────────────────────
stages = ['base', 'rrf_rerank']

emb_model_dir = Path(config.FINETUNED_EMBEDDING_MODEL)
has_emb_ft = emb_model_dir.exists() and any(emb_model_dir.iterdir())
if has_emb_ft:
    stages.append('emb_ft')
    print(f"Fine-tuned embedding model found — adding emb_ft")
else:
    print(f"Fine-tuned embedding model NOT found — skipping emb_ft")

ollama_models = subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout
has_llm_ft = config.LLM_FINETUNED_MODEL in ollama_models
if has_llm_ft:
    stages.append('llm_ft')
    print(f"Fine-tuned LLM '{config.LLM_FINETUNED_MODEL}' found — adding llm_ft")
else:
    print(f"Fine-tuned LLM '{config.LLM_FINETUNED_MODEL}' NOT found — skipping llm_ft")

if has_emb_ft and has_llm_ft:
    stages.append('full')
    print("Both fine-tuned models available — adding full stage")

stages_str = ','.join(stages)
print(f"\nRunning stages : {stages_str}")
print(f"Corpus         : {CORPUS_PATH}  (7,579 chunks)")
print(f"Eval data      : {EVAL_DATA_PATH}  (240 questions)\n")

!PYTHONUTF8=1 python scripts/14_eval_all_stages.py \
    --stages {stages_str} \
    --corpus {CORPUS_PATH} \
    --eval-data {EVAL_DATA_PATH}

print("\nAblation evaluation complete.")
print("Results saved to results/ — will be backed up in Cell 15.")

In [ ]:
# =============================================================================
# Cell 15 — Save everything to Google Drive
#
# Copies the following to My Drive/CENG493_Results/:
#   results/       — all evaluation metrics, predictions, ablation summary
#   index/         — FAISS index + metadata (for resuming without re-encoding)
#   models/        — fine-tuned LoRA adapter, merged weights, GGUF (if present)
#
# Uses dirs_exist_ok=True so re-running is safe (merges with existing files).
# Large model files (qwen25_merged/, GGUF) are only copied if they exist.
# =============================================================================

import shutil
import pathlib
import sys

sys.path.insert(0, '/content/CENG493_Project/CENG493_Project')
import config

drive_root = pathlib.Path('/content/drive/MyDrive/CENG493_Results')
drive_root.mkdir(parents=True, exist_ok=True)

# ── Results ───────────────────────────────────────────────────────────────────
results_src = config.BASE_DIR / 'results'
if results_src.exists():
    shutil.copytree(str(results_src), str(drive_root / 'results'), dirs_exist_ok=True)
    print(f"Results  → {drive_root / 'results'}")
else:
    print("No results/ directory found — run at least Cell 6 first.")

# ── FAISS index backup ────────────────────────────────────────────────────────
index_src = config.INDEX_DIR
if index_src.exists():
    shutil.copytree(str(index_src), str(drive_root / 'index_backup'), dirs_exist_ok=True)
    size_mb = sum(f.stat().st_size for f in index_src.rglob('*') if f.is_file()) / 1e6
    print(f"Index    → {drive_root / 'index_backup'}  ({size_mb:.0f} MB)")
else:
    print("No index/ directory found — run Cell 5 first.")

# ── LoRA adapter ──────────────────────────────────────────────────────────────
lora_src = config.BASE_DIR / 'models' / 'qwen25_lora'
if lora_src.exists():
    shutil.copytree(str(lora_src), str(drive_root / 'models' / 'qwen25_lora'), dirs_exist_ok=True)
    print(f"LoRA     → {drive_root / 'models' / 'qwen25_lora'}")
else:
    print("No LoRA adapter found — Cell 12 not run or did not complete.")

# ── Fine-tuned embedding model ────────────────────────────────────────────────
emb_src = pathlib.Path(config.FINETUNED_EMBEDDING_MODEL)
if emb_src.exists() and any(emb_src.iterdir()):
    dst_name = drive_root / 'models' / emb_src.name
    shutil.copytree(str(emb_src), str(dst_name), dirs_exist_ok=True)
    print(f"Emb FT   → {dst_name}")
else:
    print("No fine-tuned embedding model found — Cell 11 not run or did not complete.")

# ── GGUF model (large — only if present) ─────────────────────────────────────
gguf_src = config.BASE_DIR / 'models' / 'qwen25_merged.gguf'
if gguf_src.exists():
    size_gb = gguf_src.stat().st_size / 1e9
    print(f"GGUF ({size_gb:.1f} GB) found — copying (may take a few minutes)...")
    shutil.copy2(str(gguf_src), str(drive_root / 'models' / 'qwen25_merged.gguf'))
    print(f"GGUF     → {drive_root / 'models' / 'qwen25_merged.gguf'}")
else:
    print("No GGUF file found — Cell 13 not run or did not complete.")

print(f"\nAll available outputs saved to: {drive_root}")
print("\nSummary of Drive contents:")
for item in sorted(drive_root.rglob('*')):
    if item.is_file():
        size_mb = item.stat().st_size / 1e6
        rel = item.relative_to(drive_root)
        print(f"  {rel}  ({size_mb:.1f} MB)")
